# 00H — Check Feature Store, Registry and MLflow access


Run independent cells with the notebook's current credentials. Each cell shows
the actual SDK operation; original exceptions are left visible. A failure in one
service does not establish anything about another service. These calls use
the notebook's credentials; they do not prove managed-job role or network access.
Training/HPO, native Batch Transform and in-job MLflow logging need their own
execution checks.

Read-only discovery runs without resource names. Creation/write probes are
explicit switches and use uniquely named diagnostic resources. Paste exact role
ARNs and an approved S3 prefix before enabling the relevant probes. Do not guess
an IAM path from the STS caller ARN. No IAM, CDK or container build is performed.

Feature Store defaults to the offline S3/Glue/Athena route used by the handoff.
Set ENABLE_FEATURE_ONLINE_STORE=True before creation only when you also want
to test online GetRecord access. MLflow checks distinguish server provisioning, the UI presigned URL and
actual experiment/metric/artifact writes. A listed resource is not a successful
write test. Creation can incur charges; the final cells remove only resources
created by this kernel and retain the small S3 evidence files.



## 1. Configure independent checks

- Create clients with short connection timeouts and print the current account/Region.
- Supply exact service role ARNs and a nonempty S3 prefix for resource creation.
- Leave mutation switches false until you want to exercise that service; discovery does not need them.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from urllib.parse import urlparse
import importlib.metadata
import json
import time
import uuid
import boto3
from botocore.config import Config

approved_s3_prefix = ""
feature_role_arn = ""
tracking_role_arn = ""
output_kms_key_arn = ""
athena_workgroup = "primary"
tracking_server_name = ""  # Existing server name, or empty to name a new diagnostic server.
mlflow_version = ""  # Empty selects the service default; Describe prints the actual version.
model_image_uri = ""       # Use an actual image/checkpoint pair from 03F for the package probe.
model_data_uri = ""
CREATE_FEATURE_GROUP = False
ENABLE_FEATURE_ONLINE_STORE = False  # Offline is sufficient for this handoff.
CREATE_REGISTRY_GROUP = False
CREATE_REGISTRY_PACKAGE = False
CREATE_TRACKING_SERVER = False
WRITE_MLFLOW_RUN = False
WAIT_FOR_FEATURE_OFFLINE = False
RUN_ATHENA_QUERY = False

session = boto3.Session()
region = session.region_name
assert region, "Set the notebook's AWS Region."
config = Config(connect_timeout=5, read_timeout=20, retries={"mode": "standard", "total_max_attempts": 1})
sm = session.client("sagemaker", config=config)
fs = session.client("sagemaker-featurestore-runtime", config=config)
s3 = session.client("s3", config=config)
glue = session.client("glue", config=config)
athena = session.client("athena", config=config)
identity = session.client("sts", config=config).get_caller_identity()
suffix = uuid.uuid4().hex[:10]
diagnostic_name = "capability-" + suffix
feature_group_name = diagnostic_name
registry_group_name = diagnostic_name
selected_existing_tracking = bool(tracking_server_name)
tracking_server_name = tracking_server_name or diagnostic_name
record_id = "diagnostic-" + suffix
event_time = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")
created_feature = created_registry = created_tracking = False
package_arn = tracking_arn = query_id = mlflow_run_id = experiment_id = None
results = {"AWS caller": "confirmed"}
WORK = Path.cwd() / ".runs" / diagnostic_name
WORK.mkdir(parents=True, exist_ok=True)
print("Account / Region:", identity["Account"], region)
print("Caller:", identity["Arn"])
print("Feature group / Registry group:", diagnostic_name)
print("Tracking server:", tracking_server_name)


## 2. List the configured S3 prefix

- Use the exact approved bucket/prefix instead of choosing an unrelated first bucket.
- List at most five keys without reading their contents.
- Leave an AccessDenied or connection error visible in this cell.


In [ ]:
if approved_s3_prefix:
    location = urlparse(approved_s3_prefix.rstrip("/") + "/")
    assert location.scheme == "s3" and location.netloc and location.path.strip("/")
    bucket, prefix = location.netloc, location.path.strip("/")
    print(s3.list_objects_v2(Bucket=bucket, Prefix=prefix + "/", MaxKeys=5))
    results["S3 prefix list"] = "confirmed"
else:
    print("Set approved_s3_prefix to check S3.")


## 3. List Feature Store groups

- Call ListFeatureGroups directly.
- Print names and statuses from the first page.
- This checks discovery only; creation and record access are separate cells.


In [ ]:
response = sm.list_feature_groups(MaxResults=10)
print(json.dumps(response["FeatureGroupSummaries"], indent=2, default=str))
results["ListFeatureGroups"] = "confirmed"


## 4. Create a tiny offline feature group

- Define three synthetic fields and enable offline S3 delivery.
- Enable the online store only if ENABLE_FEATURE_ONLINE_STORE is true.
- Let Feature Store create its Glue table, exposing that permission boundary.
- Create only the unique diagnostic group; the role must persist offline records.


In [ ]:
if CREATE_FEATURE_GROUP:
    assert approved_s3_prefix and feature_role_arn, "Set the S3 prefix and exact feature_role_arn."
    feature_storage = {"S3Uri": approved_s3_prefix.rstrip("/") + "/" + diagnostic_name + "/features"}
    if output_kms_key_arn:
        feature_storage["KmsKeyId"] = output_kms_key_arn
    response = sm.create_feature_group(
        FeatureGroupName=feature_group_name,
        RecordIdentifierFeatureName="record_id", EventTimeFeatureName="event_time",
        FeatureDefinitions=[
            {"FeatureName": "record_id", "FeatureType": "String"},
            {"FeatureName": "event_time", "FeatureType": "String"},
            {"FeatureName": "value", "FeatureType": "Fractional"}],
        OnlineStoreConfig={"EnableOnlineStore": ENABLE_FEATURE_ONLINE_STORE},
        OfflineStoreConfig={"S3StorageConfig": feature_storage, "DisableGlueTableCreation": False},
        RoleArn=feature_role_arn,
        Tags=[{"Key": "Diagnostic", "Value": diagnostic_name}])
    created_feature = True
    print(response)
    results["CreateFeatureGroup"] = "confirmed; provisioning is separate"
else:
    print("Set CREATE_FEATURE_GROUP=True to exercise creation.")


## 5. Wait for feature-group provisioning

- Describe the diagnostic group for up to five minutes.
- Print the offline-store status and any failure reason.
- Group creation does not by itself establish successful offline publication.


In [ ]:
feature_description = None
if created_feature:
    deadline = time.monotonic() + 300
    while time.monotonic() < deadline:
        feature_description = sm.describe_feature_group(FeatureGroupName=feature_group_name)
        status = feature_description["FeatureGroupStatus"]
        print(status, feature_description.get("OfflineStoreStatus"), flush=True)
        if status != "Creating":
            break
        time.sleep(10)
    print(json.dumps(feature_description, indent=2, default=str))
    assert status == "Created", feature_description.get("FailureReason", status)
    results["Feature group ready"] = "confirmed"
else:
    print("Feature-group creation was not completed in this kernel.")


## 6. Write a synthetic feature record

- Call PutRecord against only this kernel's created diagnostic group.
- Write a timestamp, unique record ID and numeric value to the enabled stores.
- This tests ingestion; the next cells test readback and offline delivery.


In [ ]:
if created_feature:
    response = fs.put_record(
        FeatureGroupName=feature_group_name,
        Record=[{"FeatureName": "record_id", "ValueAsString": record_id},
                {"FeatureName": "event_time", "ValueAsString": event_time},
                {"FeatureName": "value", "ValueAsString": "7.0"}],
        TargetStores=["OfflineStore"] + (["OnlineStore"] if ENABLE_FEATURE_ONLINE_STORE else []))
    print(response)
    results["PutRecord"] = "confirmed"
else:
    print("Create the diagnostic feature group first.")


## 7. Read back the online record

- Call GetRecord only when online storage was enabled at group creation.
- Verify the returned value, not just the HTTP status.
- This is an online-store read; offline data is queried through S3/Athena below.


In [ ]:
if created_feature and ENABLE_FEATURE_ONLINE_STORE:
    response = fs.get_record(FeatureGroupName=feature_group_name, RecordIdentifierValueAsString=record_id)
    print(response)
    values = {item["FeatureName"]: item["ValueAsString"] for item in response.get("Record", [])}
    assert values.get("record_id") == record_id and float(values["value"]) == 7.0
    results["GetRecord online"] = "confirmed"
else:
    print("Online read not tested: enable the online store at creation and ingest a record first.")


## 8. Inspect offline S3 publication

- Use DescribeFeatureGroup's resolved offline S3 location.
- Optionally wait up to 15 minutes for a Parquet file; offline ingestion is asynchronous.
- Read the first bytes of a published object to test notebook S3 access separately from PutRecord.


In [ ]:
if created_feature:
    feature_description = sm.describe_feature_group(FeatureGroupName=feature_group_name)
    storage = feature_description["OfflineStoreConfig"]["S3StorageConfig"]
    offline_uri = storage.get("ResolvedOutputS3Uri", storage["S3Uri"])
    location = urlparse(offline_uri)
    print("Offline location:", offline_uri)
    deadline = time.monotonic() + (900 if WAIT_FOR_FEATURE_OFFLINE else 1)
    parquet = []
    while time.monotonic() < deadline:
        response = s3.list_objects_v2(Bucket=location.netloc, Prefix=location.path.lstrip("/"), MaxKeys=1000)
        parquet = [item for item in response.get("Contents", []) if item["Key"].endswith(".parquet")]
        if parquet or not WAIT_FOR_FEATURE_OFFLINE:
            break
        time.sleep(20)
    if parquet:
        response = s3.get_object(Bucket=location.netloc, Key=parquet[0]["Key"], Range="bytes=0-3")
        assert response["Body"].read() == b"PAR1"
        print("Readable Parquet:", parquet[0]["Key"])
        results["Offline S3 publication/read"] = "confirmed"
    else:
        print("No Parquet observed yet; not a permission denial. Rerun or enable the wait.")
else:
    print("Create and ingest the diagnostic feature group first.")


## 9. Inspect the Feature Store Glue table

- Read the actual catalog/database/table names from DescribeFeatureGroup.
- Call Glue GetTable directly.
- A Glue or Lake Formation denial remains separate from Feature Store runtime access.


In [ ]:
if created_feature:
    feature_description = sm.describe_feature_group(FeatureGroupName=feature_group_name)
    catalog = feature_description["OfflineStoreConfig"].get("DataCatalogConfig")
    assert catalog, "The feature group's Glue table is not available yet."
    # Feature Store's Catalog is an Athena name; Glue CatalogId is the AWS account ID.
    response = glue.get_table(CatalogId=identity["Account"], DatabaseName=catalog["Database"], Name=catalog["TableName"])
    print(json.dumps(response["Table"], indent=2, default=str))
    results["Glue GetTable"] = "confirmed"
else:
    print("No diagnostic feature table was created.")


## 10. Submit an offline-record query

- Query only the unique synthetic record through the generated Glue table.
- Use the configured Athena workgroup and an output prefix below this diagnostic's S3 location.
- Submission, query completion and result readback are distinct checks.


In [ ]:
if RUN_ATHENA_QUERY:
    assert created_feature and catalog, "Create the feature group and inspect its Glue table first."
    query = (f'SELECT record_id, value FROM "{catalog["Database"]}"."{catalog["TableName"]}" '
             f"WHERE record_id = '{record_id}' AND NOT is_deleted ORDER BY event_time DESC LIMIT 1")
    result_config = {"OutputLocation": approved_s3_prefix.rstrip("/") + "/" + diagnostic_name + "/athena/"}
    if output_kms_key_arn:
        result_config["EncryptionConfiguration"] = {"EncryptionOption": "SSE_KMS", "KmsKey": output_kms_key_arn}
    response = athena.start_query_execution(
        QueryString=query, WorkGroup=athena_workgroup, ResultConfiguration=result_config,
        QueryExecutionContext={"Catalog": catalog["Catalog"], "Database": catalog["Database"]})
    query_id = response["QueryExecutionId"]
    print(query_id, query)
    results["Athena StartQueryExecution"] = "confirmed"
else:
    print("Set RUN_ATHENA_QUERY=True after offline publication.")


## 11. Read and verify the offline query result

- Wait up to five minutes for this query and print its failure reason if it fails.
- Call GetQueryResults after success.
- Verify the synthetic record's numeric value so an empty query is not counted as readback.


In [ ]:
if query_id:
    deadline = time.monotonic() + 300
    while time.monotonic() < deadline:
        query_result = athena.get_query_execution(QueryExecutionId=query_id)["QueryExecution"]
        state = query_result["Status"]["State"]
        print(state, flush=True)
        if state in {"SUCCEEDED", "FAILED", "CANCELLED"}:
            break
        time.sleep(5)
    assert state == "SUCCEEDED", query_result["Status"]
    response = athena.get_query_results(QueryExecutionId=query_id)
    print(json.dumps(response["ResultSet"], indent=2))
    rows = response["ResultSet"]["Rows"][1:]
    assert rows, "No offline record returned yet; rerun after publication."
    assert rows[0]["Data"][0]["VarCharValue"] == record_id
    assert float(rows[0]["Data"][1]["VarCharValue"]) == 7.0
    results["Athena offline record readback"] = "confirmed"
else:
    print("No diagnostic query was submitted.")


## 12. List Model Registry groups

- Call ListModelPackageGroups independently of Feature Store.
- Print the first page of names and descriptions.
- No model registration or approval is implied by listing.


In [ ]:
response = sm.list_model_package_groups(MaxResults=10)
print(json.dumps(response["ModelPackageGroupSummaryList"], indent=2, default=str))
results["ListModelPackageGroups"] = "confirmed"


## 13. Create a diagnostic Registry group

- Create only this kernel's unique group if enabled.
- Keep it separate from real model package groups.
- Package registration is a separate cell and needs an actual image/checkpoint pair.


In [ ]:
if CREATE_REGISTRY_GROUP:
    response = sm.create_model_package_group(
        ModelPackageGroupName=registry_group_name,
        ModelPackageGroupDescription="Temporary capability check",
        Tags=[{"Key": "Diagnostic", "Value": diagnostic_name}])
    created_registry = True
    print(response)
    results["CreateModelPackageGroup"] = "confirmed"
else:
    print("Set CREATE_REGISTRY_GROUP=True to exercise group creation.")


## 14. Register an actual checkpoint as pending approval

- Use a successful model_image_uri/model_data_uri pair from 03F.
- Register it only in this kernel's diagnostic group as PendingManualApproval.
- Do not launch inference or approve the model automatically.


In [ ]:
if CREATE_REGISTRY_PACKAGE:
    assert created_registry and model_image_uri and model_data_uri
    response = sm.create_model_package(
        ModelPackageGroupName=registry_group_name,
        ModelApprovalStatus="PendingManualApproval",
        InferenceSpecification={
            "Containers": [{"Image": model_image_uri, "ModelDataUrl": model_data_uri}],
            "SupportedContentTypes": ["application/jsonlines"],
            "SupportedResponseMIMETypes": ["application/jsonlines"]},
        SkipModelValidation="None")
    package_arn = response["ModelPackageArn"]
    print(response)
    results["CreateModelPackage"] = "confirmed"
else:
    print("Set the image/checkpoint and CREATE_REGISTRY_PACKAGE=True to exercise registration.")


## 15. Inspect the registered package

- Describe only the package created above.
- Show its actual status, image, checkpoint and approval state.
- PendingManualApproval is intentional; registration does not approve deployment.


In [ ]:
if package_arn:
    response = sm.describe_model_package(ModelPackageName=package_arn)
    print(json.dumps({key: response.get(key) for key in [
        "ModelPackageArn", "ModelPackageStatus", "ModelApprovalStatus", "InferenceSpecification"]}, indent=2))
    results["DescribeModelPackage"] = response["ModelPackageStatus"]
else:
    print("No diagnostic package registered.")


## 16. List MLflow tracking servers

- Call ListMlflowTrackingServers independently of the other services.
- Print names, statuses and versions.
- A listed server does not establish permission to open its UI or write an experiment.


In [ ]:
response = sm.list_mlflow_tracking_servers()
print(json.dumps(response["TrackingServerSummaries"], indent=2, default=str))
results["ListMlflowTrackingServers"] = "confirmed"


## 17. Create a small diagnostic tracking server

- Use the exact tracking service role and an artifact prefix under the approved S3 location.
- Disable automatic model registration so this check does not create additional model packages.
- An existing server can be used instead by setting tracking_server_name and leaving creation off.


In [ ]:
if CREATE_TRACKING_SERVER:
    assert approved_s3_prefix and tracking_role_arn, "Set the artifact prefix and exact tracking role ARN."
    response = sm.create_mlflow_tracking_server(
        TrackingServerName=tracking_server_name, TrackingServerSize="Small",
        RoleArn=tracking_role_arn,
        ArtifactStoreUri=approved_s3_prefix.rstrip("/") + "/" + diagnostic_name + "/mlflow",
        AutomaticModelRegistration=False,
        Tags=[{"Key": "Diagnostic", "Value": diagnostic_name}],
        **({"MlflowVersion": mlflow_version} if mlflow_version else {}))
    created_tracking = True
    tracking_arn = response["TrackingServerArn"]
    print(response)
    results["CreateMlflowTrackingServer"] = "confirmed; provisioning is separate"
else:
    print("Creation disabled; the next cell can inspect the configured existing server.")


## 18. Describe the tracking server

- Call DescribeMlflowTrackingServer for the configured name.
- Capture the actual server ARN and MLflow version.
- A failure here does not prevent other-service checks; a new server can take several minutes to become ready.


In [ ]:
if created_tracking or selected_existing_tracking:
    response = sm.describe_mlflow_tracking_server(TrackingServerName=tracking_server_name)
    tracking_arn = response["TrackingServerArn"]
    print(json.dumps(response, indent=2, default=str))
    results["DescribeMlflowTrackingServer"] = response["TrackingServerStatus"]
else:
    print("Select an existing tracking_server_name or create the diagnostic server.")


## 19. Request a tracking UI session

- Call CreatePresignedMlflowTrackingServerUrl directly.
- Keep the short-lived URL in a variable instead of printing it into a screenshot.
- A generated URL establishes API permission; open it separately to test browser/network access.


In [ ]:
if created_tracking or selected_existing_tracking:
    response = sm.create_presigned_mlflow_tracking_server_url(
        TrackingServerName=tracking_server_name, ExpiresInSeconds=60, SessionExpirationDurationInSeconds=1800)
    mlflow_ui_url = response["AuthorizedUrl"]
    print("UI URL created; held in mlflow_ui_url, not printed.")
    results["CreatePresignedMlflowTrackingServerUrl"] = "confirmed; browser access not tested"
else:
    print("Select or create a tracking server first.")


## 20. Inspect MLflow client dependencies

- Print mlflow, mlflow-skinny and sagemaker-mlflow versions; either MLflow distribution can provide the client.
- Use client versions compatible with the tracking server and the approved package index.
- The following run cell leaves missing-package or authorization errors visible.


In [ ]:
for package in ["mlflow", "mlflow-skinny", "sagemaker-mlflow"]:
    try:
        print(package, importlib.metadata.version(package))
    except importlib.metadata.PackageNotFoundError:
        print(package, "not installed")


## 21. Create an experiment and a run

- Use the configured server ARN through the SageMaker MLflow client plugin.
- Create a uniquely named diagnostic experiment and run.
- This is an actual MLflow data-plane write, separate from server creation.


In [ ]:
if WRITE_MLFLOW_RUN:
    assert tracking_arn, "Describe a ready tracking server first."
    import mlflow
    mlflow.set_tracking_uri(tracking_arn)
    mlflow_client = mlflow.tracking.MlflowClient()
    experiment_id = mlflow_client.create_experiment(diagnostic_name)
    run = mlflow_client.create_run(experiment_id, tags={"purpose": "capability-check"})
    mlflow_run_id = run.info.run_id
    print("Experiment:", experiment_id, "| run:", mlflow_run_id)
    results["MLflow create experiment/run"] = "confirmed"
else:
    print("Set WRITE_MLFLOW_RUN=True to exercise experiment writes.")


## 22. Write and read a parameter and metric

- Log one parameter and one numeric metric to this diagnostic run.
- Read them back with GetRun and verify the exact values.
- This checks actual metric access rather than just successful server discovery.


In [ ]:
if mlflow_run_id:
    mlflow_client.log_param(mlflow_run_id, "source", "capability-check")
    mlflow_client.log_metric(mlflow_run_id, "probe_value", 7.0)
    run = mlflow_client.get_run(mlflow_run_id)
    assert run.data.params["source"] == "capability-check" and run.data.metrics["probe_value"] == 7.0
    print(run.data)
    results["MLflow metric/parameter write/read"] = "confirmed"
else:
    print("No diagnostic MLflow run was created.")


## 23. Write and read a small artifact

- Upload a tiny text artifact to the diagnostic run.
- Download it and verify its bytes, exposing the artifact-store permission boundary.
- Finish the run after successful verification.


In [ ]:
if mlflow_run_id:
    artifact = WORK / "probe.txt"
    artifact.write_text("artifact-access-confirmed\n")
    mlflow_client.log_artifact(mlflow_run_id, str(artifact))
    downloaded = mlflow_client.download_artifacts(mlflow_run_id, "probe.txt")
    assert Path(downloaded).read_text() == artifact.read_text()
    mlflow_client.set_terminated(mlflow_run_id)
    print("Artifact upload/download confirmed:", mlflow_run_id)
    results["MLflow artifact write/read"] = "confirmed"
else:
    print("No diagnostic MLflow run was created.")


## 24. One view to capture

- Print only operations that returned successfully and were recorded by their cells.
- Keep original exceptions beside the failed SDK calls; absence from this summary means unconfirmed.
- Save identifiers for inspection and cleanup without persisting the presigned UI URL.


In [ ]:
print("REMAINING CAPABILITIES")
print("Account / Region:", identity["Account"], region)
for name, status in results.items():
    print(name + ": " + status)
print("Unlisted operations are unconfirmed; see the original error in their cell.")
print("Feature group:", feature_group_name if created_feature else "not created")
print("Registry package:", package_arn or "not created")
print("Tracking server:", tracking_arn or "not described")
print("MLflow run:", mlflow_run_id or "not created")
(WORK / "results.json").write_text(json.dumps({
    "results": results, "feature_group": feature_group_name, "registry_group": registry_group_name,
    "package": package_arn, "tracking_server": tracking_server_name,
    "tracking_arn": tracking_arn, "mlflow_run": mlflow_run_id,
    "created_feature": created_feature, "created_registry": created_registry,
    "created_tracking": created_tracking}, indent=2) + "\n")
print("Local report:", WORK / "results.json")


## 25. Optional: remove the diagnostic feature group

- Delete only the group created by this kernel, after checking its Diagnostic tag.
- The delete call is asynchronous; the next DescribeFeatureGroup should eventually report absence.
- Offline S3 files and Glue/Athena artifacts are retained for separate inspection/removal.


In [ ]:
DELETE_FEATURE_GROUP = False
if DELETE_FEATURE_GROUP and created_feature:
    feature = sm.describe_feature_group(FeatureGroupName=feature_group_name)
    tags = {t["Key"]: t["Value"] for t in sm.list_tags(ResourceArn=feature["FeatureGroupArn"])["Tags"]}
    assert tags.get("Diagnostic") == diagnostic_name
    print(sm.delete_feature_group(FeatureGroupName=feature_group_name))
else:
    print("Feature group retained or not created by this kernel.")


## 26. Optional: remove the diagnostic Registry records

- Delete only this kernel's created package and group after checking ownership.
- Keep the underlying trained checkpoint and image intact.
- If group deletion reports it is still in use, wait for package deletion and rerun.


In [ ]:
DELETE_REGISTRY_RECORDS = False
if DELETE_REGISTRY_RECORDS and created_registry:
    group = sm.describe_model_package_group(ModelPackageGroupName=registry_group_name)
    tags = {t["Key"]: t["Value"] for t in sm.list_tags(ResourceArn=group["ModelPackageGroupArn"])["Tags"]}
    assert tags.get("Diagnostic") == diagnostic_name
    if package_arn:
        print(sm.delete_model_package(ModelPackageName=package_arn))
        package_arn = None
    print(sm.delete_model_package_group(ModelPackageGroupName=registry_group_name))
else:
    print("Registry records retained or not created by this kernel.")


## 27. Optional: remove the diagnostic tracking server

- Delete only a server created by this kernel after checking its Diagnostic tag.
- Never delete a pre-existing server selected for experiment checks.
- Artifact files remain in the configured S3 prefix; inspect or remove that diagnostic prefix separately.


In [ ]:
DELETE_TRACKING_SERVER = False
if DELETE_TRACKING_SERVER and created_tracking:
    server = sm.describe_mlflow_tracking_server(TrackingServerName=tracking_server_name)
    tags = {t["Key"]: t["Value"] for t in sm.list_tags(ResourceArn=server["TrackingServerArn"])["Tags"]}
    assert tags.get("Diagnostic") == diagnostic_name
    print(sm.delete_mlflow_tracking_server(TrackingServerName=tracking_server_name))
else:
    print("Tracking server retained; existing servers are never deleted by this notebook.")



## Access to request from the observed results

- **Now:** the approved S3 code/input/output prefix and any KMS permissions;
  the exact pipeline/job roles; Processing and native Training job access.
  Native Batch Transform additionally needs a compatible serving image and its job permissions.
- **Next:** Registry group/package creation and inspection. Keep model approval
  and production promotion with the platform's normal owners.
- **Feature Store:** group creation, service-role passing, PutRecord and the
  intended online/offline read route; offline publication also depends on S3,
  Glue, Athena/workgroup and any Lake Formation controls.
- **MLflow:** an existing server or permission to create one, the artifact-store
  service role, UI session access, `sagemaker-mlflow` data-plane permissions and
  network access. Server API access alone is insufficient.

These checks do not change the application's tracking mode or enable outbound
network calls in network-isolated training jobs. In-job live MLflow logging needs
its own permitted network route and an integration run with the actual job role.
References:
[Feature groups](https://docs.aws.amazon.com/sagemaker/latest/APIReference/API_CreateFeatureGroup.html),
[Tracking server](https://docs.aws.amazon.com/sagemaker/latest/APIReference/API_CreateMlflowTrackingServer.html),
[MLflow permissions](https://docs.aws.amazon.com/sagemaker/latest/dg/mlflow-create-tracking-server-iam.html).
